# 04 — Text Cleaning

Companion to [`../../data_cleaning/text_cleaning.md`](../../data_cleaning/text_cleaning.md).

In [ ]:
import re, unicodedata
import pandas as pd

raw = pd.Series([
    "  Loved THE product!! Highly recommend :)  ",
    "I dont like it... too expensive",
    "AMAZING!!! Worth every penny.",
    "Visit https://example.com for more, email me at user@example.com",
    "café was great",                          # accents
    "<p>HTML <b>markup</b> in the text</p>",
    "Loved THE product!! Highly recommend :)", # near-duplicate of row 0
])
raw

## 1. Decode and normalize unicode

In [ ]:
def nfc(s): return unicodedata.normalize('NFC', s)
raw.map(nfc).head()

## 2. Strip HTML

In [ ]:
def strip_html(s): return re.sub(r'<[^>]+>', ' ', s)
raw.map(strip_html).head()

## 3. Replace URLs and emails with placeholders

In [ ]:
def mask(s):
    s = re.sub(r'https?://\S+', '<URL>', s)
    s = re.sub(r'\S+@\S+', '<EMAIL>', s)
    return s
raw.map(mask).head()

## 4. Whitespace and case normalization

In [ ]:
def squash_ws(s): return re.sub(r'\s+', ' ', s).strip()
clean = raw.map(nfc).map(strip_html).map(mask).map(squash_ws)
clean

## 5. Deduplicate (case-insensitive)

In [ ]:
dedup_key = clean.str.lower()
clean[~dedup_key.duplicated()].reset_index(drop=True)

## Decisions left to you

- Lowercase or not? Depends on downstream model.
- Strip punctuation? Hurts sentiment models.
- Stop-words and stemming? Usually skip for transformer models.